In [ ]:
#Extraction de contenu d’une base de données SQL(SQLite)
#Connexion à une base de données SQL (SQLite) avec LangChain
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [5]:
#Création d’un tool personnalisé pour interroger une base SQL avec un agent
from langchain.tools import tool
@tool
def sql_query(query: str) -> str:
    """Obtain information from the database using SQL queries"""
    try:
        print(f"Executing SQL query: {query}")
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"
sql_query.invoke("SELECT * FROM Artist LIMIT 10")

Executing SQL query: SELECT * FROM Artist LIMIT 10


"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [6]:
#Création d’un agent LLM pour interroger une base SQL
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama

In [7]:
# Initialiser le modèle Ollama
model = ChatOllama(
model="llama3.2:3b", # ou mistral, gemma, etc.
)
system_prompt = """You are a SQL expert.
Rules:
- Only use sql_query tool
- The sql_query tool takes a SQL query as input and returns the result of the
query.
- Only use available columns
- If information does not exist, say so
- Do not guess
- you have to return the results in a human readable format, do not return raw
SQL results or a sql query.
Database schema:
Table Artist:
- ArtistId
- Name
"""
agent = create_agent(
model=model,
tools=[sql_query],
system_prompt=system_prompt
)

In [8]:
question = HumanMessage(content="Give me the first 5 artists in the database")
response = agent.invoke(
{"messages": [question]}
)
print(response['messages'][-1].content)

Executing SQL query: SELECT ArtistId, Name FROM Artist ORDER BY ArtistId LIMIT 5
The first five artists in the database are:

1. AC/DC - ArtistId: 1
2. Accept - ArtistId: 2
3. Aerosmith - ArtistId: 3
4. Alanis Morissette - ArtistId: 4
5. Alice In Chains - ArtistId: 5
